# 09 · Manual versus automatic perspective geometry

This separate diagnostic asks: **does correction help when the board geometry is supplied manually?** It leaves notebook 08, production decision rules and training data unchanged.

Eight fixed development photos: four missed side views and four visually near-frontal controls. Only two side views have sufficient observed physical geometry for a manual correction. The other two remain unavailable: no corners are invented. Manual markings are approximate, not calibrated pose ground truth.

1. Inspect frozen annotations and visibility labels.
2. Run or resume the comparison.
3. Compare OCR errors and the images.

Models: PP-OCRv5_mobile_det and pretrained PP-OCRv6_medium_rec, detector unclip 1.2. No lighting, rotation, optional enlargement, sharpening or super-resolution. Ground-truth visibility was selected visually before diagnostic predictions, including difficult text; there is no OCR-success visibility filter.


In [ ]:
from pathlib import Path
import json, sys, subprocess, html
from IPython.display import display, HTML, Image

cwd = Path.cwd().resolve()
candidates = [cwd / "Assignment" / "whiteboard-ocr", cwd / "whiteboard-ocr", cwd, *cwd.parents]
WB = next(p for p in candidates if (p / "scripts" / "manual_geometry_diagnostic.py").is_file())
OUT = WB / "outputs" / "manual_geometry_diagnostic_v2"
PLAN = WB / "experiments" / "manual_geometry" / "annotations.json"
RUN_OCR = False  # True runs/resumes the experiment; False opens existing results or generates previews.
print("Project:", WB)
print("Output:", OUT)


## 1. The annotations are fixed before evaluation

Red marks the upper physical horizontal frame/joint, green the lower frame, blue an observed vertical board frame. In oct17_050 the upper marking is an internal physical panel joint, assumed parallel to the lower frame; the outer top is cropped. The visible side frame supplies vertical direction without assuming the camera was held level.

The chosen ground-truth indices are held fixed across all arms. They are inherited project transcriptions, not newly verified perfect labels. Partial physical lines are excluded according to visual inspection, not model confidence.


In [ ]:
plan = json.loads(PLAN.read_text())
for case in plan["cases"]:
    print(case["photo"], "|", case["visual_role"], "|", case["manual_status"],
          "|", len(case["visible_gt_indices"]), "complete GT lines")
    print(" ", case["visibility_note"])


## 2. Run or open existing results

Use the same Python 3.12 environment as notebook 08. If RUN_OCR is False and results already exist, this cell only reads them. If True, completed photo records are resumed. A changed script, annotation, label set or environment refuses to reuse incompatible results: choose a different OUT folder for a new experiment.

**baseline:** the shared writing-region crop, line detection, ordinary quadrilateral crops and OCR.

**automatic:** the same path with the current strict perspective decision rule.

**board_roi_original:** use the manually marked writing-panel mask and keep the original perspective. This isolates board localisation from correction.

**manual_redetect:** the same manual panel mask, with geometry from the visual annotation, followed by the same detector and reader. Compare it with board_roi_original for the geometry effect. The angle/size gate is logged but deliberately bypassed for this diagnostic; text containment is still required.

**board_roi_fixed_boxes_original / manual_fixed_boxes:** take identical baseline line polygons whose centres lie on the manually marked panel, then compare their unwarped crops against their homography-transformed crops. This helps distinguish recognition effects from re-detection. No output is selected using its OCR result.


In [ ]:
if RUN_OCR or not (OUT / "results.json").exists():
    command = [sys.executable, "-u", str(WB / "scripts" / "manual_geometry_diagnostic.py"),
               "--annotations", str(PLAN), "--out", str(OUT)]
    if RUN_OCR:
        command.append("--run")
    # Stream progress and fail visibly if the comparison cannot complete.
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError("Diagnostic failed; inspect the output above before using any partial results.")
else:
    print("Using completed diagnostic results:", OUT / "results.json")


## 3. Compare results and images

CER below is uncapped corpus character error over visually selected complete labels. Missed GT lines count as deletions. It uses one-to-one text matching; stray predictions are reported separately and are not penalised in this line-matched score. This is not a full-page transcription accuracy measure.

Compare manual versus baseline **on the same two geometry-eligible photos**, not all eight baselines against two manual results. A result on familiar boards does not establish performance on new boards. Near-frontal categories are visual judgments, not measured camera poses.


In [ ]:
if (OUT / "results.json").exists():
    records = json.loads((OUT / "results.json").read_text())
    rows = []
    for r in records:
        s = r["score"]
        rows.append(f"<tr><td>{r['photo']}</td><td>{r['variant']}</td><td>{s['lines']}</td>"
                    f"<td>{s['cer']:.3%}</td><td>{s['covered']}</td><td>{s['unmatched_readings']}</td></tr>")
    display(HTML("<table border='1' cellpadding='7'><tr><th>Photo</th><th>Arm</th><th>GT lines</th>"
                 "<th>CER ↓</th><th>Covered</th><th>Unmatched readings</th></tr>" + "".join(rows) + "</table>"))
    print((OUT / "summary.md").read_text())
else:
    print("Only annotation previews exist. Set RUN_OCR=True for measurements.")
print("Interactive review:", OUT / "review.html")


In [ ]:
PHOTO = "oct17_050"  # also try oct17_142
for suffix in ("annotation", "baseline", "automatic", "board_roi_original", "manual"):
    path = OUT / f"{PHOTO}_{suffix}.jpg"
    if path.exists():
        display(HTML(f"<h4>{html.escape(PHOTO + ': ' + suffix)}</h4>"))
        display(Image(filename=str(path), width=1000))
